# CS F407 – Neural Models Lab: XOR, activations, symmetry, softmax
Redundant-sensor XOR task. Model: 2 inputs → 2 hidden → 1 output (BCE with logits). Everything is float64 and
initialised from a seeded NumPy generator so results are reproducible.

**Note:** write your own Task 1 / Task 2 answers in the report before you present this code as yours (the lab asks for this).

In [ ]:
import numpy as np, torch
import torch.nn.functional as F
torch.set_default_dtype(torch.float64)

X = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float64)
y = torch.tensor([[0],[1],[1],[0]], dtype=torch.float64)        # XOR targets
ACT = {'sigmoid': torch.sigmoid, 'tanh': torch.tanh, 'relu': torch.relu}

def make_params(seed, H=2, K=1, mode='random'):
    if mode == 'random':                       # small random init (seeded)
        r = np.random.default_rng(seed)
        W1 = r.normal(0, 1, (H, 2)); W2 = r.normal(0, 1, (K, H))
    elif mode == 'zero':
        W1 = np.zeros((H, 2)); W2 = np.zeros((K, H))
    elif mode == 'const':                      # identical non-zero weights
        W1 = np.full((H, 2), 0.5); W2 = np.full((K, H), 0.5)
    ps = [W1, np.zeros(H), W2, np.zeros(K)]    # W1, b1, W2, b2
    return [torch.tensor(p, requires_grad=True) for p in ps]

def forward(P, act):
    W1, b1, W2, b2 = P
    a1 = X @ W1.T + b1          # pre-activation of hidden layer
    h = ACT[act](a1)            # hidden activation
    z = h @ W2.T + b2           # output logits
    return a1, h, z

def train(P, act, lr=0.5, steps=5000, log=()):
    opt = torch.optim.SGD(P, lr=lr)
    hist, L0 = {}, None
    for t in range(steps + 1):
        opt.zero_grad()
        a1, h, z = forward(P, act)                            # forward pass
        loss = F.binary_cross_entropy_with_logits(z, y)       # scalar mean loss
        loss.backward()                                       # reverse-mode AD
        if L0 is None: L0 = loss.item()
        if t in log: hist[t] = (loss.item(), P[0].grad.norm().item(), P[0].detach().clone())
        if t == steps: break
        opt.step()                                            # parameter update
    return L0, loss.item(), hist

def summary(P, act):
    with torch.no_grad():
        p = torch.sigmoid(forward(P, act)[2]).squeeze()
    return p.numpy().round(4), (p > 0.5).int().numpy()

## Task 1 sanity check – a purely linear model (affine + sigmoid) cannot learn XOR

In [ ]:
torch.manual_seed(0)
lin = torch.nn.Linear(2, 1); opt = torch.optim.SGD(lin.parameters(), lr=0.5)
for _ in range(5000):
    opt.zero_grad(); l = F.binary_cross_entropy_with_logits(lin(X), y); l.backward(); opt.step()
print('linear model: loss', round(l.item(), 4), 'probs', torch.sigmoid(lin(X)).detach().squeeze().numpy().round(3))

## Task 4A – basic learning check (tanh hidden, seed 2)

In [ ]:
P = make_params(2); L0, Lf, _ = train(P, 'tanh')
probs, labels = summary(P, 'tanh')
print('initial loss', round(L0,4), '| final loss', round(Lf,5))
print('probabilities', probs, '| labels', labels, '| target', y.squeeze().int().numpy())
print('all correct:', bool((labels == y.squeeze().int().numpy()).all()))

## Task 4B – backprop check: `W1.grad` is ∂L/∂W⁽¹⁾; compare with finite differences

In [ ]:
P = make_params(0)
_, _, z = forward(P, 'tanh'); F.binary_cross_entropy_with_logits(z, y).backward()
auto = P[0].grad.clone()
num = torch.zeros_like(auto); eps = 1e-6
for i in range(2):
    for j in range(2):
        vals = []
        for s in (+eps, -eps):
            Q = make_params(0)
            with torch.no_grad(): Q[0][i, j] += s
            vals.append(F.binary_cross_entropy_with_logits(forward(Q, 'tanh')[2], y).item())
        num[i, j] = (vals[0] - vals[1]) / (2 * eps)
print('autograd\n', auto.numpy().round(4)); print('finite diff\n', num.numpy().round(4))
print('max abs difference', (auto - num).abs().max().item())

## Task 4C – symmetry experiment (zero init, and identical non-zero init)

In [ ]:
for mode in ['zero', 'const']:
    P = make_params(0, mode=mode); _, Lf, hist = train(P, 'tanh', log=(0, 1, 10, 100, 1000, 5000))
    print(f'--- init = {mode}')
    for t, (l, gn, W) in hist.items():
        print(f'step {t:5d} loss {l:.4f}  row0 {W[0].numpy().round(4)}  row1 {W[1].numpy().round(4)}')
    print('final probs', summary(P, 'tanh')[0])

## Task 4D – activation experiment (same init seed, only the activation changes)

In [ ]:
print('act      final loss   4/4 correct   ||grad W1|| at step 0')
for act in ['sigmoid', 'tanh', 'relu']:
    P = make_params(2); L0, Lf, hist = train(P, act, log=(0,))
    ok = bool((summary(P, act)[1] == y.squeeze().int().numpy()).all())
    print(f'{act:8s} {Lf:10.5f}   {str(ok):11s}   {hist[0][1]:.4f}')

print('\nRobustness over 10 seeds (successes out of 10):')
for act in ['sigmoid', 'tanh', 'relu']:
    n = 0
    for s in range(10):
        P = make_params(s); train(P, act)
        n += int((summary(P, act)[1] == y.squeeze().int().numpy()).all())
    print(f'{act:8s} {n}/10')

### Diagnosing a small gradient: saturated sigmoid vs dead ReLU (inspect pre-activations)

In [ ]:
P = make_params(4); train(P, 'relu')          # seed 4: ReLU gradient is exactly zero
a1, h, _ = forward(P, 'relu')
print('ReLU pre-activations (rows = 4 inputs):\n', a1.detach().numpy().round(3))
print('fraction of positive pre-activations:', (a1 > 0).double().mean().item())

## Task 5 – three-class extension (softmax + cross-entropy)

In [ ]:
cls = torch.tensor([0, 1, 1, 2])
def train3(seed, H=2, lr=0.5, steps=5000):
    r = np.random.default_rng(seed)
    P = [torch.tensor(r.normal(0,1,(H,2)), requires_grad=True), torch.zeros(H, requires_grad=True),
         torch.tensor(r.normal(0,1,(3,H)), requires_grad=True), torch.zeros(3, requires_grad=True)]
    opt = torch.optim.SGD(P, lr=lr)
    for t in range(steps + 1):
        opt.zero_grad(); z = torch.tanh(X @ P[0].T + P[1]) @ P[2].T + P[3]
        loss = F.cross_entropy(z, cls); loss.backward()          # softmax + CE combined
        if t == steps: break
        opt.step()
    return P, z.detach(), loss.item()

P, z, L = train3(2)
p = torch.softmax(z, dim=1)
print('W2 shape', tuple(P[2].shape), '| logits per example', z.shape[1], '| final loss', round(L, 5))
print('probabilities\n', p.numpy().round(4)); print('row sums', p.sum(1).numpy())
print('example 1 logits', z[1].numpy().round(3), '-> softmax', p[1].numpy().round(4), 'sum', p[1].sum().item())

# gradient of mean CE wrt logits should be (p - y)/N
zz = z.clone().requires_grad_(True); F.cross_entropy(zz, cls).backward()
print('max |dL/dz - (p - onehot)/N| =', (zz.grad - (p - F.one_hot(cls, 3)) / 4).abs().max().item())

# optional: shift invariance + numerical stability
naive = lambda v: torch.exp(v) / torch.exp(v).sum()
print('softmax(z+100) - softmax(z) max diff:', (torch.softmax(z[1] + 100, 0) - torch.softmax(z[1], 0)).abs().max().item())
print('naive softmax(z+1000):', naive(z[1] + 1000).numpy(), '| stable:', torch.softmax(z[1] + 1000, 0).numpy().round(4))